# OSRM Store to Polygon Distance Calculator
Calculates road distances from each store to its polygon edge points.

**Instructions:**
1. Run Cell 1 to mount Google Drive
2. Upload `store_to_points_matrix.csv` to your Google Drive (My Drive)
3. Run all remaining cells
4. Result CSV auto-downloads when done

In [ ]:
# Step 1: Mount Google Drive & load CSV
# Upload store_to_points_matrix.csv to your Google Drive first!

from google.colab import drive
drive.mount('/content/drive')

import os

# Search for the file in Google Drive
drive_root = '/content/drive/My Drive'
target = 'store_to_points_matrix.csv'
INPUT_FILE = None

# Check common locations
for folder in ['', 'Downloads', 'DS Network Design Trial']:
    path = os.path.join(drive_root, folder, target)
    if os.path.exists(path):
        INPUT_FILE = path
        break

# If not found, search recursively (top level only)
if not INPUT_FILE:
    for f in os.listdir(drive_root):
        if target.lower() in f.lower() and f.endswith('.csv'):
            INPUT_FILE = os.path.join(drive_root, f)
            break

if INPUT_FILE:
    print(f"Found file: {INPUT_FILE}")
    print(f"File size: {os.path.getsize(INPUT_FILE):,} bytes")
else:
    print("File not found! Please upload store_to_points_matrix.csv to your Google Drive root folder.")
    print("Then re-run this cell.")

In [ ]:
# Step 2: Load and preview data
import pandas as pd
import requests
import time
from tqdm.notebook import tqdm

df = pd.read_csv(INPUT_FILE)
df['store code'] = df['store code'].str.strip()
print(f"Loaded {len(df):,} rows from {df['store code'].nunique()} stores")
df.head()

In [ ]:
# Step 3: Define distance calculation functions

OSRM_BASE = "https://router.project-osrm.org"
BATCH_SIZE = 50
DELAY = 1.5
RETRY_COUNT = 3
TIMEOUT = 30


def get_batch_distances(store_lat, store_lon, point_coords):
    coords = f"{store_lon},{store_lat}"
    for plat, plon in point_coords:
        coords += f";{plon},{plat}"

    destinations = ";".join(str(i) for i in range(1, len(point_coords) + 1))
    url = f"{OSRM_BASE}/table/v1/driving/{coords}?sources=0&destinations={destinations}&annotations=distance"

    for attempt in range(RETRY_COUNT):
        try:
            resp = requests.get(url, timeout=TIMEOUT)
            if resp.status_code == 429:
                time.sleep(5 * (attempt + 1))
                continue
            data = resp.json()
            if data.get("code") == "Ok":
                return [round(d / 1000, 2) if d is not None else None for d in data["distances"][0]]
            else:
                return fallback_individual(store_lat, store_lon, point_coords)
        except Exception:
            if attempt < RETRY_COUNT - 1:
                time.sleep(2 * (attempt + 1))
            else:
                return [None] * len(point_coords)
    return [None] * len(point_coords)


def fallback_individual(store_lat, store_lon, point_coords):
    results = []
    for plat, plon in point_coords:
        url = f"{OSRM_BASE}/route/v1/driving/{store_lon},{store_lat};{plon},{plat}?overview=false"
        try:
            time.sleep(1)
            resp = requests.get(url, timeout=TIMEOUT)
            data = resp.json()
            if data.get("code") == "Ok" and data.get("routes"):
                results.append(round(data["routes"][0]["distance"] / 1000, 2))
            else:
                results.append(None)
        except Exception:
            results.append(None)
    return results


# Test connection
test = requests.get(f"{OSRM_BASE}/route/v1/driving/77.5946,12.9716;77.6200,13.0000?overview=false", timeout=10)
if test.json().get("code") == "Ok":
    print("OSRM server is reachable! Ready to go.")
else:
    print("WARNING: OSRM server issue.")

In [ ]:
# Step 4: Calculate all road distances (~3-5 minutes)

store_groups = df.groupby('store code')
distances = {}

pbar = tqdm(total=len(df), desc="Calculating distances")

for store_name, group in store_groups:
    store_lat = group['Store Latitude'].iloc[0]
    store_lon = group['Store Longitude'].iloc[0]

    polygon_edges = group['polygon edge'].tolist()
    point_lats = group['Point Latitude'].tolist()
    point_lons = group['Point Longitude'].tolist()

    for i in range(0, len(polygon_edges), BATCH_SIZE):
        batch_edges = polygon_edges[i:i + BATCH_SIZE]
        batch_coords = list(zip(point_lats[i:i + BATCH_SIZE], point_lons[i:i + BATCH_SIZE]))

        batch_distances = get_batch_distances(store_lat, store_lon, batch_coords)

        for edge, dist in zip(batch_edges, batch_distances):
            distances[edge] = dist

        pbar.update(len(batch_edges))
        time.sleep(DELAY)

pbar.close()
print(f"\nDone! Calculated {len(distances):,} distances.")

In [ ]:
# Step 5: Save results and download
from google.colab import files

df['road_distance_km'] = df['polygon edge'].map(distances)

success = df['road_distance_km'].notna().sum()
failed = df['road_distance_km'].isna().sum()
valid = df['road_distance_km'].dropna()

print(f"Results:")
print(f"  Successful: {success:,} ({success/len(df)*100:.1f}%)")
print(f"  Failed:     {failed:,} ({failed/len(df)*100:.1f}%)")
if len(valid) > 0:
    print(f"  Avg distance:  {valid.mean():.1f} km")
    print(f"  Max distance:  {valid.max():.1f} km")
    print(f"  Min distance:  {valid.min():.1f} km")
    print(f"  Median:        {valid.median():.1f} km")

OUTPUT_FILE = "store_to_polygon_with_road_distance.csv"
df.to_csv(OUTPUT_FILE, index=False)

# Also save to Google Drive
df.to_csv('/content/drive/My Drive/store_to_polygon_with_road_distance.csv', index=False)
print(f"\nSaved to Google Drive and downloading...")

files.download(OUTPUT_FILE)